Faisa Ahmed
13.02.2026

## Google Step Up Challenge: Data Science Route

This notebook prepares the environment for analysing historic campaign performance, brand lift outcomes, and creative testing results to determine:

1. Which markets and channels drove the most efficient sign-ups
2. Which campaigns generated statistically significant brand lift
3. The cost per lifted user (CPLU)
4. Which creatives resonate most with university students

## Step 0 – Environment Setup

0.1 Import Required Libraries

In [1]:
# Standard Data Manipulation
import pandas as pd
import numpy as np

# Statistical Testing
from statsmodels.stats.proportion import proportions_ztest

# Display Options (Cleaner Output)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

0.2 Load Datasets

In [2]:
# Load Data
historic_campaign = pd.read_csv("historic_campaign_data.csv")
brand_lift = pd.read_csv("brand_lift_study.csv")
creative_perf = pd.read_csv("creative_performance_report.csv")

# Confirm successful loading
print("Historic Campaign Data Shape:", historic_campaign.shape)
print("Brand Lift Study Shape:", brand_lift.shape)
print("Creative Performance Report Shape:", creative_perf.shape)


Historic Campaign Data Shape: (1664, 9)
Brand Lift Study Shape: (48, 10)
Creative Performance Report Shape: (240, 8)


0.3 Initial Data Inspection -  Preview first few rows of each dataset

In [3]:
historic_campaign.head()

,Week_Start,Week_Date,Campaign_Name,Market,Channel,Spend_USD,Impressions,Reach,Conversions
0,2023-01-02,2023-01-02,Always_On,UK,YouTube,6212,331780,111134,214
1,2023-01-02,2023-01-02,Always_On,UK,Social,4262,318639,154596,137
2,2023-01-02,2023-01-02,Always_On,UK,Display,2009,515503,101883,18
3,2023-01-02,2023-01-02,Always_On,UK,Search,5513,67707,61330,466
4,2023-01-02,2023-01-02,Always_On,DE,YouTube,5664,426034,190810,173


In [4]:
brand_lift.head()

,Campaign_Name,Market,Channel,Control_Responses,Control_Consideration,Exposed_Responses,Exposed_Consideration,Control_Rate,Exposed_Rate,Relative_Lift
0,BackToSchool_23,UK,YouTube,1824,273,2204,595,0.1497,0.2700,80.3714
1,BackToSchool_23,UK,Social,2473,370,1460,364,0.1496,0.2493,66.6368
2,BackToSchool_23,UK,Display,1741,261,1348,215,0.1499,0.1595,6.3915
3,BackToSchool_23,DE,YouTube,2236,335,1566,425,0.1498,0.2714,81.1441
4,BackToSchool_23,DE,Social,2269,340,1134,273,0.1498,0.2407,60.6590


In [5]:
creative_perf.head()

,Report_Date,Creative_Name,Market,Channel,Age_Group,Point_Est_Awareness,Point_Est_Consideration,Point_Est_Purchase_Intent
0,2026-01-15,Study With Me,UK,YouTube,18-24,6.0700,4.0000,1.6200
1,2026-01-15,Study With Me,UK,YouTube,25-34,6.2000,4.0700,1.4100
2,2026-01-15,Study With Me,UK,YouTube,35-49,5.8000,3.5300,1.3900
3,2026-01-15,Study With Me,UK,Social,18-24,4.8000,3.5200,1.5500
4,2026-01-15,Study With Me,UK,Social,25-34,5.2100,3.6100,1.6100


0.4 Data Type & Null Checks

In [6]:
def print_dataset_info(name, df):
    print("=" * 60)
    print(f"{name}".center(60))
    print("=" * 60)
    print(df.info())
    print("\n")

print_dataset_info("Historic Campaign Dataset", historic_campaign)
print_dataset_info("Brand Lift Study Dataset", brand_lift)
print_dataset_info("Creative Performance Report Dataset", creative_perf)


                 Historic Campaign Dataset                  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1664 entries, 0 to 1663
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Week_Start     1664 non-null   object
 1   Week_Date      1664 non-null   object
 2   Campaign_Name  1664 non-null   object
 3   Market         1664 non-null   object
 4   Channel        1664 non-null   object
 5   Spend_USD      1664 non-null   int64 
 6   Impressions    1664 non-null   int64 
 7   Reach          1664 non-null   int64 
 8   Conversions    1664 non-null   int64 
dtypes: int64(4), object(5)
memory usage: 117.1+ KB
None


                  Brand Lift Study Dataset                  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Campaign_Name          48 no

In [7]:
# Check for missing values
print("Historic Campaign Missing Values:\n", historic_campaign.isnull().sum())
print("\nBrand Lift Missing Values:\n", brand_lift.isnull().sum())
print("\nCreative Performance Missing Values:\n", creative_perf.isnull().sum())

Historic Campaign Missing Values:
 Week_Start       0
Week_Date        0
Campaign_Name    0
Market           0
Channel          0
Spend_USD        0
Impressions      0
Reach            0
Conversions      0
dtype: int64

Brand Lift Missing Values:
 Campaign_Name            0
Market                   0
Channel                  0
Control_Responses        0
Control_Consideration    0
Exposed_Responses        0
Exposed_Consideration    0
Control_Rate             0
Exposed_Rate             0
Relative_Lift            0
dtype: int64

Creative Performance Missing Values:
 Report_Date                  0
Creative_Name                0
Market                       0
Channel                      0
Age_Group                    0
Point_Est_Awareness          0
Point_Est_Consideration      0
Point_Est_Purchase_Intent    0
dtype: int64


0.6 Basic Data Validation Checks

In [8]:
# Ensure no negative spend or conversions
assert (historic_campaign["Spend_USD"] >= 0).all(), "Negative spend detected."
assert (historic_campaign["Conversions"] >= 0).all(), "Negative conversions detected."

# Ensure response counts are logical
assert (brand_lift["Control_Consideration"] <= brand_lift["Control_Responses"]).all()
assert (brand_lift["Exposed_Consideration"] <= brand_lift["Exposed_Responses"]).all()

print("Basic validation checks passed.")


Basic validation checks passed.


## Step 1 – Explore Historic Campaign Performance

Understand historic performance by:
1. Market
2. Channel

Identify:
- Which markets drove the most conversions
- Which channels were most efficient
- Relative cost differences across markets and platforms

Short: Where and how are campaigns most efficient historically?

1.1 Create Core Efficiency Metrics - calculate cpa, cpm, cr

In [9]:
# Create a copy to preserve raw data
campaign_df = historic_campaign.copy()

# Avoid division errors
campaign_df = campaign_df[campaign_df["Conversions"] >= 0]

# Calculate performance metrics
campaign_df["CPA"] = campaign_df["Spend_USD"] / campaign_df["Conversions"].replace(0, np.nan)
campaign_df["CPM"] = (campaign_df["Spend_USD"] / campaign_df["Impressions"]) * 1000
campaign_df["Conversion_Rate"] = campaign_df["Conversions"] / campaign_df["Reach"]

1.2 Performance by Market - Aggregate total spend and results by market.

In [10]:
print("Market Perfomance: ")
market_summary = (
    campaign_df
    .groupby("Market")
    .agg(
        Total_Spend=("Spend_USD", "sum"),
        Total_Impressions=("Impressions", "sum"),
        Total_Reach=("Reach", "sum"),
        Total_Conversions=("Conversions", "sum")
    )
    .reset_index()
)

# Recalculate aggregated efficiency metrics
market_summary["CPA"] = market_summary["Total_Spend"] / market_summary["Total_Conversions"]
market_summary["CPM"] = (market_summary["Total_Spend"] / market_summary["Total_Impressions"]) * 1000
market_summary["Conversion_Rate"] = market_summary["Total_Conversions"] / market_summary["Total_Reach"]

market_summary.sort_values("CPA")


Market Perfomance: 


,Market,Total_Spend,Total_Impressions,Total_Reach,Total_Conversions,CPA,CPM,Conversion_Rate
2,SA,2083640,172652001,55848091,338507,6.1554,12.0684,0.0061
1,EG,2151203,188418029,59479404,340022,6.3267,11.4172,0.0057
0,DE,3187902,265750160,86429705,137131,23.2471,11.9959,0.0016
3,UK,3194600,271371922,87458324,134771,23.7039,11.7720,0.0015


South Africa had the lowest CPA at $6.16, making it the most cost-efficient market. Egypt drove the highest number of conversions (340,022), indicating it’s the strongest for volume campaigns

1.3 Performance by Channel

In [11]:
print("Perfomance by channel: ")
channel_summary = (
    campaign_df
    .groupby("Channel")
    .agg(
        Total_Spend=("Spend_USD", "sum"),
        Total_Impressions=("Impressions", "sum"),
        Total_Reach=("Reach", "sum"),
        Total_Conversions=("Conversions", "sum")
    )
    .reset_index()
)

channel_summary["CPA"] = channel_summary["Total_Spend"] / channel_summary["Total_Conversions"]
channel_summary["CPM"] = (channel_summary["Total_Spend"] / channel_summary["Total_Impressions"]) * 1000
channel_summary["Conversion_Rate"] = channel_summary["Total_Conversions"] / channel_summary["Total_Reach"]

channel_summary.sort_values("CPA")


Perfomance by channel: 


,Channel,Total_Spend,Total_Impressions,Total_Reach,Total_Conversions,CPA,CPM,Conversion_Rate
1,Search,2910428,35985535,31395714,505336,5.7594,80.8777,0.0161
2,Social,2923541,201997758,81707177,205578,14.2211,14.4731,0.0025
3,YouTube,2962492,204921511,83678502,207734,14.2610,14.4567,0.0025
0,Display,1820884,455287308,92434131,31783,57.2911,3.9994,0.0003


Search channels delivered the lowest CPA at $5.76, indicating efficient targeting. Search also drove the highest number of conversions overall, suggesting it is strong for volume campaigns

1.4 Market × Channel Performance Matrix - This identifies clear winners

In [12]:
market_channel_summary = (
    campaign_df
    .groupby(["Market", "Channel"])
    .agg(
        Total_Spend=("Spend_USD", "sum"),
        Total_Reach=("Reach", "sum"),
        Total_Conversions=("Conversions", "sum")
    )
    .reset_index()
)

market_channel_summary["CPA"] = (
    market_channel_summary["Total_Spend"] /
    market_channel_summary["Total_Conversions"]
)

market_channel_summary["Conversion_Rate"] = (
    market_channel_summary["Total_Conversions"] /
    market_channel_summary["Total_Reach"]
)

market_channel_summary.sort_values("CPA")


,Market,Channel,Total_Spend,Total_Reach,Total_Conversions,CPA,Conversion_Rate
5,EG,Search,581230,6288476,180205,3.2254,0.0287
9,SA,Search,583448,6294576,180827,3.2266,0.0287
11,SA,YouTube,586347,16322642,74204,7.9018,0.0045
10,SA,Social,577913,16033091,72955,7.9215,0.0046
7,EG,YouTube,579792,16700537,73097,7.9318,0.0044
6,EG,Social,599752,16740556,74516,8.0486,0.0045
1,DE,Search,903083,9779265,74667,12.0948,0.0076
13,UK,Search,842667,9033397,69637,12.1009,0.0077
15,UK,YouTube,911210,25307489,30814,29.5713,0.0012
3,DE,YouTube,885143,25347834,29619,29.8843,0.0012


Search in EG and SA delivered the lowest CPA (~$3.23), making it extremely cost-efficient.
SA Search also drove the highest number of conversions (180,827), indicating it is the best combination of efficiency and volume.
Display channels across all markets performed poorly in both CPA and conversion rate, suggesting they are least effective for student targeting.

1.5 Identify Top Performers

In [13]:
# Most Cost-Efficient (Lowest CPA)
top_efficiency = market_channel_summary.sort_values("CPA").head(5)
top_efficiency

,Market,Channel,Total_Spend,Total_Reach,Total_Conversions,CPA,Conversion_Rate
5,EG,Search,581230,6288476,180205,3.2254,0.0287
9,SA,Search,583448,6294576,180827,3.2266,0.0287
11,SA,YouTube,586347,16322642,74204,7.9018,0.0045
10,SA,Social,577913,16033091,72955,7.9215,0.0046
7,EG,YouTube,579792,16700537,73097,7.9318,0.0044


The top 5 most cost-efficient combinations are dominated by Search channels in EG and SA. YouTube and Social appear next, but at more than double the CPA, indicating less efficiency

In [14]:
# Highest Conversion Volume
top_volume = market_channel_summary.sort_values("Total_Conversions", ascending=False).head(5)
top_volume

,Market,Channel,Total_Spend,Total_Reach,Total_Conversions,CPA,Conversion_Rate
9,SA,Search,583448,6294576,180827,3.2266,0.0287
5,EG,Search,581230,6288476,180205,3.2254,0.0287
1,DE,Search,903083,9779265,74667,12.0948,0.0076
6,EG,Social,599752,16740556,74516,8.0486,0.0045
11,SA,YouTube,586347,16322642,74204,7.9018,0.0045


Search in SA and EG dominates both efficiency and volume. DE Search contributes moderately to volume but is less cost-efficient. YouTube and Social in SA and EG are moderate performers, indicating they may be secondary options for scale campaigns.

## Step 2 – Statistical Significance Testing (Brand Lift Study)

Determine which campaigns actually influenced student consideration

Use a two-proportion Z-test to test whether the difference between Control and Exposed groups is statistically significant

Keep only campaigns with p < 0.05 for further analysis

2.1 Import Required Packages

In [15]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest


2.2 Load Brand Lift Study Data

In [16]:
brand_lift = pd.read_csv("brand_lift_study.csv")

# Inspect the first rows
brand_lift.head()


,Campaign_Name,Market,Channel,Control_Responses,Control_Consideration,Exposed_Responses,Exposed_Consideration,Control_Rate,Exposed_Rate,Relative_Lift
0,BackToSchool_23,UK,YouTube,1824,273,2204,595,0.1497,0.2700,80.3714
1,BackToSchool_23,UK,Social,2473,370,1460,364,0.1496,0.2493,66.6368
2,BackToSchool_23,UK,Display,1741,261,1348,215,0.1499,0.1595,6.3915
3,BackToSchool_23,DE,YouTube,2236,335,1566,425,0.1498,0.2714,81.1441
4,BackToSchool_23,DE,Social,2269,340,1134,273,0.1498,0.2407,60.6590


2.3 Define Function to Run Z-Test on Each Row

In [17]:
def test_statistical_significance(row):
    # Counts of "successes" (consideration) in Control and Exposed
    counts = [row['Control_Consideration'], row['Exposed_Consideration']]
    # Total sample sizes
    nobs = [row['Control_Responses'], row['Exposed_Responses']]
    # Run two-proportion Z-test
    stat, pval = proportions_ztest(counts, nobs)
    # Determine significance
    significant = pval < 0.05
    return pd.Series({'z_stat': stat, 'p_value': pval, 'Significant': significant})


2.4 Apply Function to Brand Lift Data

In [18]:
brand_lift_stats = brand_lift.apply(test_statistical_significance, axis=1)

# Merge the results back into original dataframe
brand_lift = pd.concat([brand_lift, brand_lift_stats], axis=1)

# Inspect
brand_lift.head()


,Campaign_Name,Market,Channel,Control_Responses,Control_Consideration,Exposed_Responses,Exposed_Consideration,Control_Rate,Exposed_Rate,Relative_Lift,z_stat,p_value,Significant
0,BackToSchool_23,UK,YouTube,1824,273,2204,595,0.1497,0.2700,80.3714,-9.2427,0.0000,True
1,BackToSchool_23,UK,Social,2473,370,1460,364,0.1496,0.2493,66.6368,-7.7533,0.0000,True
2,BackToSchool_23,UK,Display,1741,261,1348,215,0.1499,0.1595,6.3915,-0.7315,0.4645,False
3,BackToSchool_23,DE,YouTube,2236,335,1566,425,0.1498,0.2714,81.1441,-9.2253,0.0000,True
4,BackToSchool_23,DE,Social,2269,340,1134,273,0.1498,0.2407,60.6590,-6.5037,0.0000,True


2.5 Filter Only Statistically Significant Campaigns

In [19]:
brand_lift_sig = brand_lift[brand_lift['Significant'] == True].reset_index(drop=True)

# How many campaigns remain?
print(f"Number of statistically significant campaigns: {brand_lift_sig.shape[0]}")

Number of statistically significant campaigns: 40


In [20]:
brand_lift_sig[['Campaign_Name',
                'Market',
                'Channel',
                'Relative_Lift',
                'p_value']].head(10)


,Campaign_Name,Market,Channel,Relative_Lift,p_value
0,BackToSchool_23,UK,YouTube,80.3714,0.0000
1,BackToSchool_23,UK,Social,66.6368,0.0000
2,BackToSchool_23,DE,YouTube,81.1441,0.0000
3,BackToSchool_23,DE,Social,60.6590,0.0000
4,BackToSchool_23,SA,YouTube,103.3268,0.0000
5,BackToSchool_23,SA,Social,99.8949,0.0000
6,BackToSchool_23,SA,Display,18.4277,0.0376
7,BackToSchool_23,EG,YouTube,102.4991,0.0000
8,BackToSchool_23,EG,Social,84.5058,0.0000
9,BackToSchool_23,EG,Display,22.6446,0.0070


## Step 3 – Join Datasets & Calculate Cost Per Lifted User (CPLU)

Combine Historic Campaign Data with statistically significant Brand Lift Study

Calculate the number of students influenced and CPLU

Identify the most efficient campaigns for Google

3.1 Prepare for the Join

In [21]:
# Create join keys for clarity
historic_campaign['join_key'] = (
    historic_campaign['Campaign_Name'].astype(str) + "_" +
    historic_campaign['Market'].astype(str) + "_" +
    historic_campaign['Channel'].astype(str)
)

brand_lift_sig['join_key'] = (
    brand_lift_sig['Campaign_Name'].astype(str) + "_" +
    brand_lift_sig['Market'].astype(str) + "_" +
    brand_lift_sig['Channel'].astype(str)
)

3.2 Merge Historic Campaign Data with Brand Lift

In [22]:
# Left join historic data with significant brand lift
campaign_lift = pd.merge(
    historic_campaign,
    brand_lift_sig,
    on='join_key',
    how='inner',  # only keep campaigns with significant lift
    suffixes=('_historic', '_lift')
)

# Check for missing values in key columns
campaign_lift[['Campaign_Name_historic', 'Market_historic', 'Channel_historic', 'Relative_Lift', 'Spend_USD', 'Reach']].head()

,Campaign_Name_historic,Market_historic,Channel_historic,Relative_Lift,Spend_USD,Reach
0,ExamPrep_23,UK,YouTube,86.8415,17643,575804
1,ExamPrep_23,UK,YouTube,86.8415,23466,710943
2,ExamPrep_23,UK,YouTube,86.8415,16723,484635
3,ExamPrep_23,UK,YouTube,86.8415,12439,443907
4,ExamPrep_23,UK,YouTube,86.8415,22215,614402


3.3 Calculate Lifted Users

In [23]:
# Lifted Users = Reach * Absolute Lift
campaign_lift['Lifted_Users'] = campaign_lift['Reach'] * campaign_lift['Relative_Lift']

# Optional: replace 0 Lifted Users with NaN to avoid division errors
campaign_lift['Lifted_Users'].replace(0, np.nan, inplace=True)


3.4 Calculate Cost Per Lifted User (CPLU)

In [24]:
# CPLU = Spend / Lifted Users
campaign_lift['CPLU'] = campaign_lift['Spend_USD'] / campaign_lift['Lifted_Users']

3.5 Rank Campaigns by Efficiency

In [25]:
# Rank campaigns: lower CPLU = more efficient
campaign_lift_sorted = campaign_lift.sort_values('CPLU').reset_index(drop=True)

# Display top 10 most efficient campaigns
campaign_lift_sorted[['Campaign_Name_historic','Market_historic','Channel_historic','Spend_USD','Reach','Relative_Lift','Lifted_Users','CPLU']].head(10)


,Campaign_Name_historic,Market_historic,Channel_historic,Spend_USD,Reach,Relative_Lift,Lifted_Users,CPLU
0,ExamPrep_23,SA,YouTube,11326,491300,98.9910,"48,634,262.0871",0.0002
1,BackToSchool_23,EG,YouTube,15038,581130,102.4991,"59,565,285.1302",0.0003
2,BackToSchool_23,SA,Social,11211,433939,99.8949,"43,348,286.9056",0.0003
3,BackToSchool_23,EG,YouTube,13477,494186,102.4991,"50,653,605.9012",0.0003
4,BackToSchool_24,DE,YouTube,20140,821046,88.8123,"72,919,015.5470",0.0003
5,BackToSchool_24,SA,YouTube,8255,279089,103.9312,"29,006,050.3788",0.0003
6,NewYear_24,SA,Social,12901,443370,101.8487,"45,156,661.5773",0.0003
7,BackToSchool_23,SA,YouTube,13093,438091,103.3268,"45,266,544.0740",0.0003
8,NewYear_24,EG,Social,10973,365377,103.4868,"37,811,710.5175",0.0003
9,BackToSchool_23,SA,Social,8074,276660,99.8949,"27,636,919.1414",0.0003


In [26]:
# Top 5 campaigns
top_5_cplu = campaign_lift.sort_values('CPLU').reset_index(drop=True)

top_5_cplu = top_5_cplu[
    ['Campaign_Name_historic',
     'Market_historic',
     'Channel_historic',
     'Lifted_Users',
     'CPLU']
].head(5)

top_5_cplu['Lifted_Users'] = (top_5_cplu['Lifted_Users'] / 1_000_000).round(1)
top_5_cplu['CPLU'] = top_5_cplu['CPLU'].round(4)

top_5_cplu

,Campaign_Name_historic,Market_historic,Channel_historic,Lifted_Users,CPLU
0,ExamPrep_23,SA,YouTube,48.6000,0.0002
1,BackToSchool_23,EG,YouTube,59.6000,0.0003
2,BackToSchool_23,SA,Social,43.3000,0.0003
3,BackToSchool_23,EG,YouTube,50.7000,0.0003
4,BackToSchool_24,DE,YouTube,72.9000,0.0003


Insights for Slide

1. Most Cost-Efficient Campaign 

ExamPrep_23 in SA on YouTube had the lowest CPLU ($0.0002 per lifted user), making it the most cost-efficient campaign for influencing student consideration.

2. Highest Reach / Volume Campaign

BackToSchool_24 in DE on YouTube reached the highest number of students (~72.9M), indicating strong potential for large-scale impact even at slightly higher CPLU ($0.0003).

3. Market & Channel Pattern

YouTube campaigns consistently outperform Social in terms of both efficiency (CPLU) and total students influenced, suggesting YouTube is the optimal channel across multiple markets.

## Step 4 – Creative Performance Analysis

- Identify which creative themes resonate most with students
- Focus on Consideration Lift
- Check differences by Market, Channel, and Age Group (18–24 key)
- Output top creatives for Google’s campaign

4.1 Filter for Key Age Group (18–24)

In [27]:
# Focus on main target audienc
creative_18_24 = creative_perf[creative_perf['Age_Group'] == "18-24"].copy()
# Check the shape
print(f"Number of creative tests for 18-24: {creative_18_24.shape[0]}")


Number of creative tests for 18-24: 80


4.2 Identify Top Creatives by Consideration Lift

In [33]:
# Sort creatives by highest consideration lift
top_creatives = creative_18_24.sort_values(
    'Point_Est_Consideration',
    ascending=False
).reset_index(drop=True)

# Select relevant columns
top_creatives_clean = top_creatives[[
    'Creative_Name',
    'Market',
    'Channel',
    'Point_Est_Consideration'
]].head(5)

# Rename columns to be presentation-friendly
top_creatives_clean = top_creatives_clean.rename(columns={
    'Creative_Name': 'Creative',
    'Market': 'Market',
    'Channel': 'Channel',
    'Point_Est_Consideration': 'Consideration Lift (%)'
})

# Round for cleaner display
top_creatives_clean['Consideration Lift (%)'] = (
    top_creatives_clean['Consideration Lift (%)']
    .round(2)
)

top_creatives_clean


,Creative,Market,Channel,Consideration Lift (%)
0,Life Hack,DE,Social,21.8300
1,Life Hack,DE,YouTube,19.9700
2,Life Hack,UK,YouTube,19.2400
3,Life Hack,UK,Social,19.0400
4,Life Hack,SA,YouTube,13.9500


The creative ‘Life Hack’ achieved the highest Consideration Lift among 18–24 students, with peak performance in Germany via Social (21.8%). Its strong and consistent lift across multiple markets (UK, SA, EG) indicates it resonates exceptionally well with the student audience, making it the ideal theme to prioritize in the upcoming campaign